In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import math

In [5]:
df = pd.read_excel("wuenic_input_to_pdf_desktop.xlsx", sheet_name="wuenic_master")
df.head()

,Country,ISOCountryCode,Vaccine,Year,WUENIC,WUENICPreviousRevision,GradeOfConfidence,AdministrativeCoverage,GovernmentEstimate,ChildrenVaccinated,ChildrenInTarget,BirthsUNPD,SurvivingInfantsUNPD,Comment
0,Afghanistan,AFG,BCG,1997,43,43.0,2,43.0,NaN,NaN,NaN,961674,896754,Legacy estimate. GoC=R+
1,Afghanistan,AFG,BCG,1998,35,35.0,2,NaN,35.0,NaN,NaN,985909,920265,Estimate informed by reported data. EPI Covera...
2,Afghanistan,AFG,BCG,1999,38,38.0,2,43.0,38.0,381008.0,878443.0,1013252,947416,Estimate informed by reported data. Afghanista...
3,Afghanistan,AFG,BCG,2000,30,30.0,2,38.0,30.0,412754.0,1079435.0,1036525,970393,Estimate informed by reported data. Data refle...
4,Afghanistan,AFG,BCG,2001,43,43.0,2,66.0,43.0,494628.0,747690.0,1006562,943488,Estimate informed by reported data. GoC=R+ D+


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import norm

YEAR = 2025

VACCINE_NAMES = {
    "BCG": "Tuberculosis (BCG)",
    "DTP1": "DTP, 1st dose",
    "DTP3": "DTP, 3rd dose",
    "HepB3": "Hepatitis B, 3rd dose",
    "HepBB": "Hepatitis B, birth dose",
    "Hib3": "Hib, 3rd dose",
    "IPV1": "Polio (IPV), 1st dose",
    "IPVC": "Polio (IPV), final dose",
    "MCV1": "Measles, 1st dose",
    "MCV2": "Measles, 2nd dose",
    "MENGA": "Meningitis A",
    "PCVC": "Pneumococcal (PCV), final dose",
    "RCV1": "Rubella, 1st dose",
    "RotaC": "Rotavirus, final dose",
    "YFV": "Yellow fever",
}

# sequential blue ramp: light -> dark ("more coverage = darker")
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE = "#c3c2b7"

seq_steps = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
seq_cmap = LinearSegmentedColormap.from_list("seq_blue", seq_steps)

# one distribution per vaccine, latest year
sub = df[(df["Year"] == YEAR) & df["WUENIC"].notna()]
stats = (
    sub.groupby("Vaccine")["WUENIC"]
    .agg(median="median", n="count")
    .sort_values("median", ascending=False)
)
vaccines = stats.index.tolist()
n_vax = len(vaccines)
med_min, med_max = stats["median"].min(), stats["median"].max()

# fixed-bandwidth KDE (percentage points) so peak sharpness is comparable across rows
BW = 3.5
x_grid = np.linspace(0, 100, 400)

def kde(values, grid, bw=BW):
    v = np.asarray(values)[:, None]
    g = grid[None, :]
    return norm.pdf(g, loc=v, scale=bw).mean(axis=0)

densities = {v: kde(sub.loc[sub["Vaccine"] == v, "WUENIC"].values, x_grid) for v in vaccines}
global_max_density = max(d.max() for d in densities.values())

ROW_GAP = 1.0
PEAK_HEIGHT = 0.86 * ROW_GAP
height_scale = PEAK_HEIGHT / global_max_density

fig, ax = plt.subplots(figsize=(9.5, 0.56 * n_vax + 1.4), dpi=150)
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)

baselines = {v: (n_vax - 1 - i) * ROW_GAP for i, v in enumerate(vaccines)}

for v in vaccines:
    y0 = baselines[v]
    dens = densities[v]
    norm_val = (stats.loc[v, "median"] - med_min) / (med_max - med_min) if med_max > med_min else 0.5
    color = seq_cmap(0.15 + 0.85 * norm_val)
    ax.fill_between(x_grid, y0, y0 + dens * height_scale, color=color, edgecolor=SURFACE, linewidth=1.4)

# 90% coverage reference line
ax.axvline(90, color=INK_MUTED, linewidth=1, linestyle=(0, (3, 3)), zorder=5)
ax.text(90, (n_vax - 1) * ROW_GAP + PEAK_HEIGHT + 0.35, "90% benchmark",
        ha="center", va="bottom", fontsize=9, color=INK_MUTED)

# direct labels: vaccine + full name (left), median % and n (right)
for v in vaccines:
    y0 = baselines[v]
    med = stats.loc[v, "median"]
    n_countries = int(stats.loc[v, "n"])
    ax.text(-3, y0 + 0.10, v, ha="right", va="bottom", fontsize=10.5, fontweight="bold", color=INK_PRIMARY)
    ax.text(-3, y0 - 0.06, VACCINE_NAMES.get(v, ""), ha="right", va="top", fontsize=8.2, color=INK_SECONDARY)
    ax.text(103, y0 + 0.10, f"{med:.0f}%", ha="left", va="bottom", fontsize=10.5, fontweight="bold", color=INK_PRIMARY)
    ax.text(103, y0 - 0.06, f"n={n_countries}", ha="left", va="top", fontsize=8.2, color=INK_MUTED)

ax.set_xlim(-32, 118)
ax.set_ylim(-0.3, (n_vax - 1) * ROW_GAP + PEAK_HEIGHT + 0.9)

ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.set_xticklabels(["0%", "20%", "40%", "60%", "80%", "100%"], color=INK_MUTED, fontsize=9)
for x in [0, 20, 40, 60, 80, 100]:
    ax.axvline(x, color=GRIDLINE, linewidth=1, zorder=0)

ax.set_yticks([])
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(BASELINE)
ax.tick_params(axis="x", length=0)

ax.text(-32, (n_vax - 1) * ROW_GAP + PEAK_HEIGHT + 1.55,
        "How does vaccine coverage vary across countries?",
        ha="left", va="bottom", fontsize=15, fontweight="bold", color=INK_PRIMARY)
ax.text(-32, (n_vax - 1) * ROW_GAP + PEAK_HEIGHT + 0.95,
        f"Distribution of national coverage estimates (WUENIC, {YEAR}) across reporting countries, one ridge per vaccine.\n"
        "Ordered by median coverage; darker blue = higher coverage. Labels show the median and number of countries.",
        ha="left", va="bottom", fontsize=9.2, color=INK_SECONDARY)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
TREND_VACCINES = ["DTP3", "MCV2"]
TREND_VACCINE_NAMES = {"DTP3": "DTP, 3rd dose", "MCV2": "Measles, 2nd dose"}
YEARS = list(range(2005, 2026))  # MCV2 has too few reporting countries before this
n_yrs = len(YEARS)

# precompute per (vaccine, year): values, median, n, density
trend_data = {}
for v in TREND_VACCINES:
    for y in YEARS:
        vals = df.loc[(df["Vaccine"] == v) & (df["Year"] == y) & df["WUENIC"].notna(), "WUENIC"].values
        trend_data[(v, y)] = {
            "n": len(vals),
            "median": np.median(vals) if len(vals) else np.nan,
            "density": kde(vals, x_grid) if len(vals) else np.zeros_like(x_grid),
        }

all_medians = [d["median"] for d in trend_data.values() if not np.isnan(d["median"])]
med_min, med_max = min(all_medians), max(all_medians)
global_max_density = max(d["density"].max() for d in trend_data.values())

ROW_GAP = 1.0
PEAK_HEIGHT = 0.86 * ROW_GAP
height_scale = PEAK_HEIGHT / global_max_density

baselines = {y: i * ROW_GAP for i, y in enumerate(YEARS)}  # oldest at bottom, newest at top
top_y = (n_yrs - 1) * ROW_GAP

fig, axes = plt.subplots(1, 2, figsize=(14.5, 0.5 * n_yrs + 2.1), dpi=150)
fig.patch.set_facecolor(SURFACE)

for ax, v in zip(axes, TREND_VACCINES):
    ax.set_facecolor(SURFACE)

    for y in YEARS:
        y0 = baselines[y]
        d = trend_data[(v, y)]
        if d["n"] == 0:
            continue
        norm_val = (d["median"] - med_min) / (med_max - med_min) if med_max > med_min else 0.5
        color = seq_cmap(0.15 + 0.85 * norm_val)
        ax.fill_between(x_grid, y0, y0 + d["density"] * height_scale,
                         color=color, edgecolor=SURFACE, linewidth=1.3)

    ax.axvline(90, color=INK_MUTED, linewidth=1, linestyle=(0, (3, 3)), zorder=5)
    ax.text(90, top_y + PEAK_HEIGHT + 0.30, "90%", ha="center", va="bottom",
            fontsize=8.5, color=INK_MUTED)

    for y in YEARS:
        y0 = baselines[y]
        d = trend_data[(v, y)]
        ax.text(-3, y0 + 0.08, str(y), ha="right", va="bottom",
                fontsize=8.6, fontweight="bold", color=INK_PRIMARY)
        if d["n"] > 0:
            ax.text(103, y0 + 0.10, f"{d['median']:.0f}%", ha="left", va="bottom",
                    fontsize=8.6, fontweight="bold", color=INK_PRIMARY)
            ax.text(103, y0 - 0.06, f"n={d['n']}", ha="left", va="top",
                    fontsize=7.2, color=INK_MUTED)

    ax.set_xlim(-16, 122)
    ax.set_ylim(-0.3, top_y + PEAK_HEIGHT + 0.75)
    ax.set_xticks([0, 20, 40, 60, 80, 100])
    ax.set_xticklabels(["0%", "20%", "40%", "60%", "80%", "100%"], color=INK_MUTED, fontsize=8.5)
    for x in [0, 20, 40, 60, 80, 100]:
        ax.axvline(x, color=GRIDLINE, linewidth=1, zorder=0)
    ax.set_yticks([])
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(BASELINE)
    ax.tick_params(axis="x", length=0)

    ax.set_title(f"{v} — {TREND_VACCINE_NAMES[v]}", loc="left", fontsize=12.5,
                 fontweight="bold", color=INK_PRIMARY, pad=14)

fig.suptitle("How has coverage changed over time?", x=0.065, y=1.015,
             ha="left", fontsize=16, fontweight="bold", color=INK_PRIMARY)
fig.text(0.065, 0.975,
          "Distribution of national coverage estimates (WUENIC) across reporting countries, one ridge per year.\n"
          "Newest year on top; darker blue = higher coverage. Labels show the median and number of reporting countries.",
          ha="left", va="top", fontsize=9.5, color=INK_SECONDARY)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()